In [0]:
from pyspark.sql.functions import *

customers_df = spark.table(
    "cicd_dev.silver.customers"
)

sales_df = spark.table(
    "cicd_dev.silver.sales"
)

In [0]:
customer_analytics = (
    customers_df.alias("c")
    .join(
        sales_df.alias("s"),
        "customer_id",
        "left"
    )
    .groupBy(
        "customer_id",
        "customer_name",
        "city"
    )
    .agg(
        count("sale_id").alias("total_orders"),
        sum("sales_amount").alias("total_sales")
    )
)

In [0]:
customer_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "cicd_dev.gold.gold_customer_analytics"
    )

In [0]:
%sql
SELECT *
FROM cicd_dev.gold.gold_customer_analytics
ORDER BY total_sales DESC;

In [0]:
from pyspark.sql.functions import current_timestamp

record_count = customer_analytics.count()

audit_df = spark.createDataFrame(
    [
        (
            "gold_customer_analytics",
            record_count,
            "SUCCESS"
        )
    ],
    ["table_name","record_count","status"]
).select(
    "*",
    current_timestamp().alias("load_time")
)

audit_df.write.mode("append").saveAsTable(
    "cicd_dev.gold.gold_audit"
)

In [0]:
%sql
SELECT *
FROM cicd_dev.gold.gold_audit;